# 📓 Maker-Collection Indexierung in Qdrant

**Autor:** Sakina Ahmadi
**Beschreibung:** Dieses Notebook indexiert alle 3.734 Marker-geparsten Papers
in einer neuen Qdrant-Collection (`maker_hierarchical_1024`).

## Ziel
1. **Collection erstellen** – mit HNSW-Index und Cosine Similarity
2. **Alle Maker-Papers laden** – aus dem Kaggle-Dataset
3. **Hierarchisches Chunking** – 1024 Tokens, 10% Overlap
4. **Embedding mit mxbai-large** – 1024 Dimensionen
5. **Batch-Upload zu Qdrant** – 64 Chunks pro Batch

---
## 1. Installation & Setup

Wir installieren die benötigten Pakete und klonen das GitHub-Repository
mit den Modulen (`chunker.py`, `embedder.py`).

In [ ]:
# =====================================================================
# 1. DEPENDENCIES INSTALLIEREN
# =====================================================================
!pip install pandas qdrant-client tqdm langchain-community langchain-core transformers

In [ ]:
# =====================================================================
# 2. GITHUB-REPOSITORY KLONEN
# =====================================================================
!git clone https://github.com/Sakinashmadi87/AutoML_Maker.git
%cd AutoML_Maker

---
## 2. Importe & Qdrant-Client

Wir importieren alle benötigten Bibliotheken und stellen die Verbindung
zu Qdrant Cloud her.

**Wichtig:** Die Secrets `QDRANT_URL_M` und `QDRANT_API_KEY_M` müssen
in Kaggle hinterlegt sein.

In [ ]:
# =====================================================================
# 3. IMPORTS & QDRANT CLIENT
# =====================================================================
import uuid
import gc
from pathlib import Path
from tqdm import tqdm

from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct
from kaggle_secrets import UserSecretsClient
from qdrant_client.models import VectorParams, Distance, HnswConfigDiff

# Module aus dem geklonten Repository
from modules.chunker import chunk_hierarchical
from modules.embedder import embed_texts

# ==========================================
# 4. QDRANT CLIENT INITIALISIEREN
# ==========================================
user_secrets = UserSecretsClient()
url = user_secrets.get_secret("QDRANT_URL_M")
api_key = user_secrets.get_secret("QDRANT_API_KEY_M")

client = QdrantClient(url=url, api_key=api_key)

COLLECTION = "maker_hierarchical_1024"
EMBED_MODEL = "mxbai-large"

print(f"🌐 Verbunden mit Qdrant Cloud")
print(f"📦 Collection: {COLLECTION}")
print(f"🧠 Embedding: {EMBED_MODEL}")

---
## 3. Collection erstellen

Wir erstellen eine neue Qdrant-Collection mit:
- **1024 Dimensionen** (für mxbai-large)
- **Cosine Similarity** als Distanzmetrik
- **HNSW-Index** mit m=16 und ef_construct=200

In [ ]:
# =====================================================================
# 5. COLLECTION ERSTELLEN
# =====================================================================
client.create_collection(
    collection_name="maker_hierarchical_1024",
    vectors_config=VectorParams(
        size=1024,
        distance=Distance.COSINE
    ),
    hnsw_config=HnswConfigDiff(
        m=16,
        ef_construct=200
    )
)

print("🏗️ Neue Collection erstellt: maker_hierarchical_1024")

---
## 4. Alle Maker-Papers laden

Wir laden alle 3.734 Marker-geparsten Papers aus dem Kaggle-Dataset.
Jedes Paper liegt in einem eigenen Unterordner mit einer `.md`-Datei.

In [ ]:
# =====================================================================
# 6. ALLE MAKER-PAPERS LADEN
# =====================================================================
MARKER_DIR = Path("/kaggle/input/datasets/sakinaahmadi/marker-parsed-papers-3734/marker_parsed_papers")

all_parsed = {
    d.name for d in MARKER_DIR.iterdir()
    if d.is_dir() and (d / f"{d.name}.md").exists()
}

print(f"📚 Gesamt geparst: {len(all_parsed)} Papers")

---
## 5. Indexierung aller Papers

Für jedes Paper:
1. **Markdown-Datei lesen**
2. **Hierarchisches Chunking** – 1024 Tokens, 10% Overlap
3. **Embedding mit mxbai-large** – 1024 Dimensionen
4. **Batch-Upload zu Qdrant** – 64 Chunks pro Batch

**Batch-Größen:**
- 20 Papers pro Embedding-Batch
- 64 Chunks pro Upsert-Batch

In [ ]:
# =====================================================================
# 7. INDEXIERUNG ALLER PAPERS
# =====================================================================
BATCH_SIZE_PAPERS = 20
UPSERT_BATCH = 64

current_chunks = []

for idx, paper_id in enumerate(sorted(all_parsed), start=1):

    md_path = MARKER_DIR / paper_id / f"{paper_id}.md"
    text = md_path.read_text(encoding="utf-8")

    # Chunking
    chunks = chunk_hierarchical(text, chunk_size=1024, overlap=102)

    for c in chunks:
        if len(c.strip()) > 20:
            current_chunks.append({
                "text": c,
                "paper_id": paper_id
            })

    # Wenn Batch voll → Embedding + Upsert
    if idx % BATCH_SIZE_PAPERS == 0 or idx == len(all_parsed):

        if current_chunks:  # ➕ Sicherheitscheck: Leere Batches überspringen
            texts = [c["text"] for c in current_chunks]
            embeddings = embed_texts(texts, model_key=EMBED_MODEL, is_query=False)

            points = []
            for i, item in enumerate(current_chunks):
                points.append(
                    PointStruct(
                        id=str(uuid.uuid4()),
                        vector=embeddings[i],
                        payload={
                            "paper_id": item["paper_id"],
                            "text_llm": item["text"],
                            "chunk_method": "hierarchical",
                            "chunk_size": 1024
                        }
                    )
                )

            # Upsert in kleinen Batches
            for k in range(0, len(points), UPSERT_BATCH):
                client.upsert(
                    collection_name=COLLECTION,
                    points=points[k:k+UPSERT_BATCH]
                )

            print(f"💾 Fortschritt: {idx}/{len(all_parsed)} Papers indexiert (+{len(points)} Chunks)")

            current_chunks.clear()
            gc.collect()
        else:
            print(f"⚠️ Batch {idx} enthält keine gültigen Chunks (übersprungen)")

print("\n🏆 Maker-Collection vollständig neu indexiert!")

---
## 6. Zusammenfassung

### Was wurde gemacht?
- **Collection:** `maker_hierarchical_1024`
- **Parser:** Marker
- **Chunking:** Hierarchisch (1024 Tokens, 10% Overlap)
- **Embedding:** mxbai-large (1024 Dimensionen)
- **Index:** HNSW (m=16, ef_construct=200)
- **Distanz:** Cosine Similarity

### Nächste Schritte
1. Retrieval-Evaluierung mit dieser Collection durchführen
2. Mit Docling- und PyMuPDF4LLM-Collections vergleichen
3. AutoML-Optimierung der Parameter